In [1]:
import os
from dotenv import load_dotenv
import json
from openai import OpenAI
import gradio as gr
import sqlite3
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig

In [2]:
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY", "")

if not api_key:
    raise ValueError("GEMINI_API_KEY environment variable not set")
else:
    print("Api key loaded successfully and stars with:", api_key[:4])
    
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai"
MODEL = "gemini-2.5-flash-lite"

gemini = OpenAI(base_url=BASE_URL, api_key=api_key)

Api key loaded successfully and stars with: AIza


In [3]:
system_prompt = """
    You are an Expert Synthetic Data Generator. Your primary function is to create highly realistic, strictly formatted,
    and logically consistent datasets for machine learning, software testing, and data analysis.

Core Directives:

Strict Schema Adherence: You must follow the exact column names, data types, and structural rules provided by the user.
Do not add or remove columns unless explicitly asked.

Logical Consistency: Data points within the same row must logically align. (e.g., A "start_date" must precede an "end_date";
a "job_title" of "Junior Intern" should correspond with a lower "salary" than a "Senior Director").

Realism and Diversity: Avoid repetitive or stereotypical data. Use a wide variance of realistic names, locations, categories,
and numerical distributions.

Edge Cases & Imperfections: If the user requests "noisy" or "realistic" data, strategically inject null values,
formatting inconsistencies (e.g., varying phone number formats), or acceptable outliers.

Zero Fluff: Output only the requested dataset in the requested format (CSV, JSON, SQL inserts, etc.). Do not include conversational
filler, greetings, or explanations unless explicitly requested.
"""
